# Notebook 10b - Training RGB Thresholds from ceilometer

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
import matplotlib.pyplot as plt
import xarray as xr

In [7]:
path = '/storage/cdalden/goes/colorado/goes16/cloud_counts/'
file_template = 'goes16_cloud_frequency_colorado_{yearmonth}.nc'
datasets = {}
# Only load August 2022 data for analysis
files = ['202201', '202202', '202205', '202207', '202208', '202301', '202302', '202304', '202305']
for file in files:
    datasets[file] = xr.open_dataset(
        path + file_template.format(yearmonth=file),
        chunks={'t': 100}  # Add chunking for lazy loading
    )
    print('opened {i}'.format(i=file))

if len(files) > 1:
    cloud_ds = xr.concat([datasets[file] for file in files], dim='t', combine_attrs='override')
else:
    cloud_ds = datasets[files[0]].copy()  # Use .copy() to avoid reference issues
    
cloud_ds = cloud_ds.rename({'t': 'time'})

# Cleanup: close individual datasets to free memory
for ds in datasets.values():
    ds.close()
del datasets


cloud_ds = cloud_ds.sel(latitude=38.958, longitude=-106.989, method='nearest')
cloud_ds

opened 202201
opened 202202
opened 202205
opened 202207
opened 202208
opened 202301
opened 202302
opened 202304
opened 202305


<xarray.Dataset> Size: 783kB
Dimensions:    (time: 48960)
Coordinates:
  * time       (time) datetime64[ns] 392kB 2022-10-01T00:02:30 ... 2023-05-31...
    latitude   float64 8B 38.96
    longitude  float64 8B -107.0
Data variables:
    clouds     (time) int64 392kB dask.array<chunksize=(100,), meta=np.ndarray>

In [8]:
start_time = pd.to_datetime('2022-01-01')
end_time = pd.to_datetime('2023-05-31')

In [9]:

# Select time range and filter after 14Z
cloud_ds = cloud_ds.sel(
    time=cloud_ds['time'].where(
        ((cloud_ds['time'] > start_time) &
        (cloud_ds['time'] < end_time) &
        (cloud_ds['time'].dt.hour >= 14))
    ).dropna('time')
)


In [10]:
met_ds = xr.open_dataset('./met_30min.nc', engine='h5netcdf')
met_ds



<xarray.Dataset> Size: 3MB
Dimensions:                    (time: 31200)
Coordinates:
  * time                       (time) datetime64[ns] 250kB 2021-08-31T18:00:0...
Data variables: (12/20)
    org_precip_accum           (time) float32 125kB ...
    pwd_cumul_rain             (time) float32 125kB ...
    pwd_cumul_snow             (time) float32 125kB ...
    tbrg_precip_total          (time) float32 125kB ...
    tbrg_precip_total_corr     (time) float32 125kB ...
    atmos_pressure             (time) float32 125kB ...
    ...                         ...
    wdir_vec_mean              (time) float64 250kB ...
    pwd_err_code               (time) float64 250kB ...
    pwd_mean_vis_1min          (time) float64 250kB ...
    lat                        (time) float32 125kB ...
    lon                        (time) float32 125kB ...
    alt                        (time) float32 125kB ...
Attributes:
    units:      mm
    long_name:  Original Precipitation Accumulation

In [11]:
path = '/storage/cdalden/goes/surface_obs/gucceilM1.b1/'
file = 'sail_guc_ceilometer_2021_2023.nc'
ceil_ds = xr.open_dataset(
    path + file,
    chunks={'time': 500}  # Add chunking for lazy loading
)
# Select time range and filter between 8am local and 6pm local (MDT) in one step
ceil_ds = ceil_ds.sel(
    time=ceil_ds['time'].where(
        ((ceil_ds['time'] > start_time) &
        (ceil_ds['time'] < end_time) &
        (ceil_ds['time'].dt.hour >= 14))
    ).dropna('time')
)
ceil_binary = (~ceil_ds['first_cbh'].isnull()).astype(int)

# Close original dataset - we only need the derived ceil_binary
ceil_ds.close()
del ceil_ds

In [12]:
ceil_binary

<xarray.DataArray 'first_cbh' (time: 346440)> Size: 3MB
dask.array<astype, shape=(346440,), dtype=int64, chunksize=(499,), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 3MB 2022-01-01T14:00:03 ... 2022-08-31T23:...
Attributes:
    long_name:            Lowest cloud base height detected
    units:                m
    valid_min:            0.0
    valid_max:            7700.0
    ancillary_variables:  qc_first_cbh detection_status
    comment:              If detection_status is 1, 2 or 3, a lowest cloud ba...

In [13]:
# 1) Remove duplicate times if present
_, uniq_idx = np.unique(ceil_binary['time'], return_index=True)
ceil_binary = ceil_binary.isel(time=np.sort(uniq_idx))

# 2) Build offset 5-min target times (offset = 2.5 minutes = 150 seconds)
t0 = pd.to_datetime(ceil_binary['time'].values.min())
t1 = pd.to_datetime(ceil_binary['time'].values.max())

# Start at first 5-min multiple after t0, then add 2.5 min offset
# choose an anchored start: round up to nearest 5min boundary then add offset
start_5min = (t0.ceil('5min') + pd.Timedelta(seconds=150))
target_times = pd.date_range(start=start_5min, end=t1, freq='5min')

# 3) Pick the nearest original sample to each target time (tolerance = 2.5 min)
tol = pd.Timedelta(seconds=150)

# Use a loop-based approach to robustly select nearest times
indices = []
for t in target_times:
    # Find nearest time in ceil_binary
    time_diff = np.abs(ceil_binary['time'].values.astype('datetime64[ns]') - np.datetime64(t, 'ns'))
    idx = np.argmin(time_diff)
    # Only include if within tolerance
    if time_diff[idx] <= tol.value:
        indices.append(idx)

if indices:
    ceil_binary_5min_offset = ceil_binary.isel(time=indices)
else:
    # Fallback: create empty DataArray with target times
    ceil_binary_5min_offset = xr.DataArray(
        np.zeros(len(target_times)), 
        dims='time', 
        coords={'time': ('time', target_times)}
    )

# 4) Quick checks
print("target times (first 6):", target_times[:6])
print("Result count:", ceil_binary_5min_offset.sizes['time'])
print("Sum (number of 1s):", int(ceil_binary_5min_offset.compute().sum().item()))

# Replace original variable if desired
ceil_binary = ceil_binary_5min_offset

target times (first 6): DatetimeIndex(['2022-01-01 14:07:30', '2022-01-01 14:12:30',
               '2022-01-01 14:17:30', '2022-01-01 14:22:30',
               '2022-01-01 14:27:30', '2022-01-01 14:32:30'],
              dtype='datetime64[ns]', freq='5min')
Result count: 18489
Sum (number of 1s): 8558


In [14]:

# Compute cloud_ds to load into memory
cloud_ds_computed = cloud_ds.compute()

# Select the nearest ceilometer binary for each cloud_ds timestep
ceil_aligned = ceil_binary.sel(time=cloud_ds_computed['time'], method='nearest')

# Extract cloud frequency as a binary (threshold at 50% for cloud detection)
# Assuming cloud frequency values are 0-100 or 0-1
cloud_frequency = cloud_ds_computed['clouds'].squeeze()
cloud_binary = (cloud_frequency > 0.5).astype(int)

# Align both binaries to common times (use ceil_aligned index)
ceil_aligned_values = ceil_aligned.values
cloud_binary_values = cloud_binary.values

print(f"Cloud DS size: {np.round(cloud_ds_computed.nbytes/10000,1)} mb")
print(f"Ceilometer aligned: {len(ceil_aligned_values)} timesteps")
print(f"Cloud binary: {len(cloud_binary_values)} timesteps")

# Cleanup
del cloud_ds_computed

Cloud DS size: 52.0 mb
Ceilometer aligned: 32520 timesteps
Cloud binary: 32520 timesteps


In [15]:

# Compare the two binaries with evaluation metrics
# Ceilometer is treated as ground truth, Cloud DS is the model
print("=== Cloud DS Model vs Ceilometer Truth ===\n")

# Calculate performance metrics (y_true=ceilometer, y_pred=cloud_ds)
accuracy = accuracy_score(ceil_aligned_values, cloud_binary_values)
precision = precision_score(ceil_aligned_values, cloud_binary_values, zero_division=0)
recall = recall_score(ceil_aligned_values, cloud_binary_values, zero_division=0)
f1 = f1_score(ceil_aligned_values, cloud_binary_values, zero_division=0)

# Print the metrics
print("Performance Metrics:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

# Detailed classification report
print("\nClassification Report:")
print(classification_report(ceil_aligned_values, cloud_binary_values, target_names=['No Cloud', 'Cloud']))

# Confusion matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(ceil_aligned_values, cloud_binary_values)
print("\nConfusion Matrix (Rows=Truth/Ceilometer, Cols=Prediction/CloudDS):")
print(f"                Cloud DS Pred")
print(f"           No Cloud  Cloud")
print(f"Ceil No Cl {cm[0,0]:6d}  {cm[0,1]:6d}")
print(f"Ceil Cloud {cm[1,0]:6d}  {cm[1,1]:6d}")

# Calculate agreement statistics
agreement = np.sum(cloud_binary_values == ceil_aligned_values) / len(ceil_aligned_values)
disagreement = 1 - agreement
print(f"\nOverall Agreement: {agreement:.4f}")
print(f"Overall Disagreement: {disagreement:.4f}")


=== Cloud DS Model vs Ceilometer Truth ===

Performance Metrics:
Accuracy:  0.7023
Precision: 0.3426
Recall:    0.6684
F1 Score:  0.4530

Classification Report:
              precision    recall  f1-score   support

    No Cloud       0.90      0.71      0.80     26524
       Cloud       0.34      0.67      0.45      5996

    accuracy                           0.70     32520
   macro avg       0.62      0.69      0.62     32520
weighted avg       0.80      0.70      0.73     32520


Confusion Matrix (Rows=Truth/Ceilometer, Cols=Prediction/CloudDS):
                Cloud DS Pred
           No Cloud  Cloud
Ceil No Cl  18832    7692
Ceil Cloud   1988    4008

Overall Agreement: 0.7023
Overall Disagreement: 0.2977


In [16]:

# Compute F1 score and confusion matrix for each month
from sklearn.metrics import confusion_matrix, f1_score
import pandas as pd

# Get month from ceil_aligned time coordinates
months = pd.to_datetime(ceil_aligned.time.values).to_period('M').unique()

print("=== Monthly Performance Metrics ===\n")

for month in months:
    # Filter data for this month
    month_mask = (pd.to_datetime(ceil_aligned.time.values).to_period('M') == month)
    
    # Get indices for this month
    month_indices = np.where(month_mask)[0]
    
    if len(month_indices) > 0:
        y_true_month = ceil_aligned_values[month_indices]
        y_pred_month = cloud_binary_values[month_indices]
        
        # Calculate F1 score
        f1_month = f1_score(y_true_month, y_pred_month, zero_division=0)
        
        # Confusion matrix
        cm_month = confusion_matrix(y_true_month, y_pred_month)
        
        print(f"Month: {month}")
        print(f"F1 Score: {f1_month:.4f}")
        print(f"Confusion Matrix:")
        print(f"           Cloud DS Pred")
        print(f"      No Cloud  Cloud")
        print(f"Ceil No Cl {cm_month[0,0]:6d}  {cm_month[0,1]:6d}")
        print(f"Ceil Cloud {cm_month[1,0]:6d}  {cm_month[1,1]:6d}")
        print()


=== Monthly Performance Metrics ===

Month: 2022-08
F1 Score: 0.2567
Confusion Matrix:
           Cloud DS Pred
      No Cloud  Cloud
Ceil No Cl  12602    6511
Ceil Cloud    855    1272

Month: 2022-01
F1 Score: 0.2407
Confusion Matrix:
           Cloud DS Pred
      No Cloud  Cloud
Ceil No Cl   2773     816
Ceil Cloud    105     146

Month: 2022-05
F1 Score: 0.8028
Confusion Matrix:
           Cloud DS Pred
      No Cloud  Cloud
Ceil No Cl   1826     265
Ceil Cloud    359    1270

Month: 2022-07
F1 Score: 0.7744
Confusion Matrix:
           Cloud DS Pred
      No Cloud  Cloud
Ceil No Cl   1631     100
Ceil Cloud    669    1320



In [1]:
import pandas as pd

In [2]:
path = '/storage/cdalden/goes/washington/goes17/rgb_composite/'
file = 'fhl_camera_goes_pixel_20220713_20220930.csv'

df = pd.read_csv(path + file)
df

,green,blue,red,clouds
0,0.060023,0.000574,0.0,b
1,0.060833,0.007203,0.0,b
2,0.055563,0.009413,0.0,b
3,0.054347,0.000000,0.0,b
4,0.053942,0.000000,0.0,b
...,...,...,...,...
4773,0.128683,0.351769,0.0,b
4774,0.152290,0.413954,0.0,b
4775,0.145995,0.426818,0.0,b
4776,0.112152,0.287412,0.0,b
